# Customer Churn Analysis and Model Training

This notebook inspects the actual Telco data, produces focused exploratory analysis, and runs the reproducible training pipeline.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from train import load_and_clean_data

raw_data = pd.read_csv(ROOT / 'data' / 'WA_Fn-UseC_-Telco-Customer-Churn.csv')
raw_data.head()

In [ ]:
print('Shape:', raw_data.shape)
print('Columns:', raw_data.columns.tolist())
raw_data.info()
display(raw_data.isna().sum().to_frame('missing_values'))
print('Duplicate rows:', raw_data.duplicated().sum())
display(raw_data.describe(include='all').T)
print('Numerical:', raw_data.select_dtypes(include='number').columns.tolist())
print('Categorical:', raw_data.select_dtypes(exclude='number').columns.tolist())
print('Target:', 'Churn')

In [ ]:
data = load_and_clean_data(ROOT / 'data' / 'WA_Fn-UseC_-Telco-Customer-Churn.csv')
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
plot_specs = [('Churn', axes[0,0]), ('Contract', axes[0,1]), ('PaymentMethod', axes[0,2]), ('InternetService', axes[1,0]), ('SeniorCitizen', axes[1,1]), ('OnlineSecurity', axes[1,2])]
for column, axis in plot_specs:
    if column == 'Churn':
        data['Churn'].map({0: 'No Churn', 1: 'Churn'}).value_counts().plot(kind='bar', ax=axis, color=['steelblue', 'tomato'])
    else:
        pd.crosstab(data[column], data['Churn'], normalize='index').plot(kind='bar', stacked=True, ax=axis, colormap='coolwarm')
    axis.set_title(f'Churn by {column}')
    axis.set_ylabel('Customer share')
    axis.legend(title='Churn')
plt.tight_layout()
plt.show()

# Observe which groups have a larger red (churn) share; month-to-month contracts and electronic check users commonly warrant closer investigation.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for churn_value, label in [(0, 'No Churn'), (1, 'Churn')]:
    subset = data.loc[data['Churn'] == churn_value]
    axes[0].hist(subset['tenure'], bins=18, alpha=.6, label=label)
    axes[1].hist(subset['MonthlyCharges'], bins=18, alpha=.6, label=label)
axes[0].set(title='Tenure by Churn', xlabel='Months', ylabel='Customers')
axes[1].set(title='Monthly Charges by Churn', xlabel='Monthly Charges', ylabel='Customers')
for axis in axes: axis.legend()
plt.tight_layout()
plt.show()

# Compare distributions rather than treating correlation as causation. Shorter-tenure and higher-charge segments may show more churn.

## Train and evaluate

Run the next cell to tune Logistic Regression, train the DNN, save the preprocessing pipeline and models, and write actual evaluation results to `reports/`.

In [ ]:
from train import main
main()
pd.read_csv(ROOT / 'reports' / 'model_comparison.csv')